# Neural Networks for Images — Exercises — STUDENT VERSION

These exercises accompany the lecture slides *"Neural Networks for Images"* (K. Reluga) and let you implement, in Python, the main ideas covered there:

1. **2D convolution (cross-correlation) from scratch** — the core CNN operation
2. **Pooling layers** — max/average pooling and (approximate) translation invariance
3. **A LeNet-style CNN** trained on MNIST — the classic conv + pool + FC pattern
4. **Transfer learning** with a pretrained ResNet18 — reusing a modern architecture
5. **Neural style transfer** — content loss + Gram-matrix style loss, "inverting" a CNN to generate an image

Each exercise has a short recap of the relevant slide content, followed by a code cell with `TODO`s for you to fill in, and a test/demo cell that uses your implementation.

**Note on compute**: exercises 1-2 are pure NumPy and run instantly. Exercises 3-5 use PyTorch; a GPU speeds things up but is *not* required — all training loops use small models/subsets so that they finish in a few minutes on a CPU.


## 0. Setup: required libraries

Before running this notebook, make sure the following libraries are installed:

- `numpy`
- `matplotlib`
- `scipy`
- `scikit-image` (for sample images, package name `scikit-image`, import name `skimage`)
- `torch` (PyTorch)
- `torchvision`

You can install them all at once with:

```bash
pip install numpy matplotlib scipy scikit-image torch torchvision
```

(If you don't have a GPU, the default CPU build of `torch`/`torchvision` from the command above is all you need.)

The cell below **checks** which of these libraries are already available in your Python environment, and prints the exact `pip install` command for anything that is missing, so you don't have to install libraries you already have.


In [ ]:
import importlib

# Maps the name used in `import ...` to the name used in `pip install ...`
# (they differ for scikit-image and Pillow, for example)
required_libraries = {
    "numpy": "numpy",
    "matplotlib": "matplotlib",
    "scipy": "scipy",
    "skimage": "scikit-image",
    "torch": "torch",
    "torchvision": "torchvision",
}

missing = []
for import_name, pip_name in required_libraries.items():
    try:
        importlib.import_module(import_name)
        print(f"[OK]      {import_name:<12s} is installed.")
    except ImportError:
        print(f"[MISSING] {import_name:<12s} is NOT installed.")
        missing.append(pip_name)

print()
if missing:
    print("Some libraries are missing. Install them by running this command")
    print("in a terminal (or in a notebook cell, prefixed with '!'):\n")
    print(f"    pip install {' '.join(missing)}")
else:
    print("All required libraries are installed -- you are ready to go!")


## Common imports

Run this cell once at the start; all exercises rely on it.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline
np.random.seed(0)


### A note on SSL / dataset downloads (macOS)

Exercises 3-5 download data or pretrained weights over HTTPS (MNIST, CIFAR-10, VGG19 weights). On some Python installations -- most commonly the **python.org installer on macOS** -- Python ships without access to your system's root certificates, and any HTTPS download fails with:

```
SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate
```

The cell below fixes this **once, for the whole notebook**, by pointing Python's default SSL context at the certificate bundle from the `certifi` package (installed automatically as a dependency of `requests`/other libraries; install it directly with `pip install certifi` if needed). This is the standard fix and does not disable certificate verification -- it just tells Python where to find a valid, up-to-date bundle of certificates.

(If you are on macOS and installed Python from python.org, you can alternatively fix this system-wide by running the `Install Certificates.command` script found in your `/Applications/Python 3.x/` folder -- but the cell below works regardless of how Python was installed.)


In [ ]:
import ssl
import certifi

# Point Python's default HTTPS context at certifi's certificate bundle.
# Fixes "CERTIFICATE_VERIFY_FAILED" errors when downloading MNIST/CIFAR-10/
# pretrained weights later in the notebook (common on macOS).
ssl._create_default_https_context = lambda: ssl.create_default_context(cafile=certifi.where())
print("SSL default context now uses certifi bundle at:", certifi.where())


---
## Exercise 1 — 2D convolution (cross-correlation) from scratch

**Recap (slides):** deep learning "convolution" is really **cross-correlation**:

$$[\mathbf{W} \circledast \mathbf{X}](i,j) = \sum_{u=0}^{H-1}\sum_{v=0}^{W-1} w_{u,v}\, x_{i+u,j+v}$$

A small filter (kernel) $\mathbf{W}$ slides over the image $\mathbf{X}$ and, at each position, computes an inner product between the filter and the local image patch (**template matching**). With zero-padding $p$ and stride $s$, the output size is

$$\left\lfloor \frac{x_h + 2p - f_h + s}{s} \right\rfloor \times \left\lfloor \frac{x_w + 2p - f_w + s}{s} \right\rfloor$$

**Your task:** implement `conv2d(image, kernel, stride=1, padding=0)` that computes this operation on a single-channel (grayscale) image.


In [ ]:
import numpy as np


def conv2d(image, kernel, stride=1, padding=0):
    """
    2D cross-correlation ("convolution" in the deep learning sense).

    Parameters
    ----------
    image : np.ndarray, shape (H, W)
    kernel : np.ndarray, shape (fh, fw)
    stride : int
    padding : int, amount of zero-padding added on every side of the image

    Returns
    -------
    output : np.ndarray, shape (out_h, out_w), the feature map
    """
    # TODO 1: if padding > 0, pad `image` with zeros on every side
    #         (hint: np.pad)

    # TODO 2: compute the output height/width `out_h`, `out_w` using the
    #         formula from the slides (with the given stride and padding)

    # TODO 3: allocate the output array and fill it in by sliding the
    #         kernel over the (padded) image with the given stride,
    #         computing the sum of the elementwise product at each position

    raise NotImplementedError("Implement conv2d")


**Test / demo** (given): we build a small synthetic image containing a diagonal edge, and a $3\times3$ diagonal-edge detector (matched filter), analogous to the *"2d convolution as template matching"* slide. We check your function against `scipy.signal.correlate2d` and visualize the resulting feature map ("heat map").

In [ ]:
from scipy import signal

# A small synthetic image with a diagonal line (a 'template' the filter should detect)
image = np.zeros((10, 10))
for k in range(10):
    if k < 9:
        image[k, k] = 1.0
        image[k, k + 1] = 1.0

# A filter tuned to detect diagonal lines sloping down-and-to-the-right
kernel = np.array([
    [ 1, -1, -1],
    [-1,  1, -1],
    [-1, -1,  1],
])

my_output = conv2d(image, kernel, stride=1, padding=0)
ref_output = signal.correlate2d(image, kernel, mode="valid")

print("Output shape:", my_output.shape, " (expected:", ref_output.shape, ")")
print("Matches scipy.signal.correlate2d:", np.allclose(my_output, ref_output))

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(image, cmap="gray")
axes[0].set_title("Input image")
axes[1].imshow(my_output, cmap="hot")
axes[1].set_title("Feature map (your conv2d)")
for ax in axes:
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
plt.show()

# Sanity-check the output-size formula with padding/stride
p, s = 1, 2
out = conv2d(image, kernel, stride=s, padding=p)
expected_h = (image.shape[0] + 2 * p - kernel.shape[0] + s) // s
expected_w = (image.shape[1] + 2 * p - kernel.shape[1] + s) // s
print(f"With padding={p}, stride={s}: got shape {out.shape}, formula predicts ({expected_h}, {expected_w})")


---
## Exercise 2 — Pooling and (approximate) translation invariance

**Recap (slides):** pooling summarizes a local window of the feature map into a single value, giving (approximate) invariance to *where* exactly a pattern occurs:

- **Max pooling**: take the maximum over a local window
- **Average pooling**: take the mean over a local window

**Your task:** implement `max_pool2d(feature_map, pool_size=2, stride=2)` and `avg_pool2d(feature_map, pool_size=2, stride=2)` (no padding, single channel).


In [ ]:
def max_pool2d(feature_map, pool_size=2, stride=2):
    """Max pooling over a single-channel feature map (no padding)."""
    # TODO: compute out_h, out_w (same formula as conv2d with padding=0),
    #       then fill in the output by taking the max over each pool_size x
    #       pool_size window, moving by `stride` at a time.
    raise NotImplementedError("Implement max_pool2d")


def avg_pool2d(feature_map, pool_size=2, stride=2):
    """Average pooling over a single-channel feature map (no padding)."""
    # TODO: same as max_pool2d but take the mean of each window instead
    #       of the max.
    raise NotImplementedError("Implement avg_pool2d")


**Test / demo** (given): recall the *"lack of translation invariance"* slide — a plain matched filter gives a very different response depending on where the object sits. Here we compare the **raw feature map** to the **pooled feature map** for two versions of the image that differ only by a 1-pixel shift, and check that pooling makes the summary statistic more stable to the shift.

In [ ]:
# Same diagonal-edge image as before, plus a version shifted by 1 pixel
image_shifted = np.zeros((10, 10))
for k in range(10):
    if k < 8:
        image_shifted[k, k + 1] = 1.0
        image_shifted[k, k + 2] = 1.0

fmap1 = conv2d(image, kernel, stride=1, padding=0)
fmap2 = conv2d(image_shifted, kernel, stride=1, padding=0)

pooled1 = max_pool2d(fmap1, pool_size=2, stride=2)
pooled2 = max_pool2d(fmap2, pool_size=2, stride=2)

raw_diff = np.mean(np.abs(fmap1 - fmap2))
pooled_diff = np.mean(np.abs(pooled1 - pooled2))

print(f"Mean abs. difference, raw feature maps  : {raw_diff:.3f}")
print(f"Mean abs. difference, pooled feature maps: {pooled_diff:.3f}")
print("Pooling reduced the sensitivity to the 1-pixel shift:", pooled_diff <= raw_diff)

fig, axes = plt.subplots(2, 2, figsize=(7, 7))
axes[0, 0].imshow(fmap1, cmap="hot"); axes[0, 0].set_title("Feature map (original)")
axes[0, 1].imshow(fmap2, cmap="hot"); axes[0, 1].set_title("Feature map (shifted)")
axes[1, 0].imshow(pooled1, cmap="hot"); axes[1, 0].set_title("Max-pooled (original)")
axes[1, 1].imshow(pooled2, cmap="hot"); axes[1, 1].set_title("Max-pooled (shifted)")
for ax in axes.ravel():
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
plt.show()


---
## Exercise 3 — A LeNet-style CNN for MNIST digit classification

**Recap (slides):** a *basic CNN architecture* alternates convolutional layers with max-pooling layers, and ends with (global-average-pooling or flatten +) fully-connected layers and a softmax classifier — the pattern used by **LeNet** (1998), the earliest successful CNN for digit recognition.

**Your task:** complete the `SimpleLeNet` class below, following this architecture (input images are $28\times28$, single channel):

| Layer | Details |
|---|---|
| `conv1` | 1 → 6 channels, kernel size 5, padding 2 (keeps size $28\times28$) |
| `pool1` | max pool, kernel size 2, stride 2 → $14\times14$ |
| `conv2` | 6 → 16 channels, kernel size 5, no padding → $10\times10$ |
| `pool2` | max pool, kernel size 2, stride 2 → $5\times5$ |
| `fc1`   | $16\times5\times5$ → 120 |
| `fc2`   | 120 → 84 |
| `fc3`   | 84 → `num_classes` |

Apply a ReLU nonlinearity after every layer except the last one.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class SimpleLeNet(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        # TODO: define self.conv1, self.pool1, self.conv2, self.pool2,
        #       self.fc1, self.fc2, self.fc3 according to the table above
        #       (use nn.Conv2d, nn.MaxPool2d, nn.Linear)
        raise NotImplementedError("Define the layers")

    def forward(self, x):
        # TODO: chain the layers defined in __init__, with a ReLU after
        #       every layer except the last (self.fc3). Don't forget to
        #       flatten the tensor (x.view(x.size(0), -1)) before fc1.
        raise NotImplementedError("Implement the forward pass")


**Training / evaluation** (given): we load MNIST via `torchvision`, train on a subset for a couple of epochs (fast on CPU), and report test accuracy. Compare this to the slide's claim of *98.8% after just 1 epoch* on the full dataset — with our small subset/short training you should still comfortably beat a random baseline (10%) and typically reach ~90%+.

In [ ]:
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

transform = transforms.ToTensor()

# One of torchvision's default MNIST mirrors (yann.lecun.com) has been
# unreliable/offline; make sure the working S3 mirror is tried first.
datasets.MNIST.mirrors = [
    "https://ossci-datasets.s3.amazonaws.com/mnist/",
] + [m for m in datasets.MNIST.mirrors if "ossci-datasets" not in m]

train_full = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_full = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

# Use a subset for a fast, CPU-friendly demo. Increase these for higher accuracy.
train_subset = Subset(train_full, range(5000))
test_subset = Subset(test_full, range(1000))

train_loader = DataLoader(train_subset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_subset, batch_size=256, shuffle=False)

model = SimpleLeNet(num_classes=10).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

n_epochs = 3
for epoch in range(n_epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
    print(f"Epoch {epoch + 1}/{n_epochs} - train loss: {running_loss / len(train_subset):.4f}")

model.eval()
correct, total = 0, 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        preds = model(images).argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

print(f"\nTest accuracy on {total} held-out images: {100 * correct / total:.2f}%")


---
## Exercise 4 — Transfer learning with a pretrained ResNet18

**Recap (slides):** modern CNN architectures such as **ResNet** use residual blocks with skip connections, $\mathbf{x}_{l+1} = \varphi(\mathbf{x}_l + \mathcal{F}_l(\mathbf{x}_l))$, to enable training much deeper networks. Rather than training such a network from scratch, we can reuse one **pretrained** on ImageNet as a generic feature extractor, and only train a new classification head for our task (**transfer learning**) — much cheaper, and effective even with little data.

**Your task:** complete `build_transfer_model(num_classes)`:
1. load a pretrained `torchvision.models.resnet18`
2. freeze **all** its parameters (`requires_grad = False`)
3. replace the final fully-connected layer (`model.fc`) with a **new**, trainable `nn.Linear` layer mapping to `num_classes` outputs


In [ ]:
import torchvision.models as models


def build_transfer_model(num_classes):
    """
    Build a transfer-learning model:
      - pretrained ResNet18 backbone, frozen
      - a new trainable linear classification head with `num_classes` outputs
    """
    # TODO 1: load a pretrained ResNet18
    #         (models.resnet18(weights=models.ResNet18_Weights.DEFAULT))

    # TODO 2: freeze every parameter of the model (loop over
    #         model.parameters() and set param.requires_grad = False)

    # TODO 3: replace model.fc with a new nn.Linear(in_features, num_classes)
    #         layer. Use model.fc.in_features for the input size.
    #         (a freshly created layer has requires_grad=True by default)

    raise NotImplementedError("Build the transfer-learning model")


**Training / evaluation** (given): we fine-tune the new head on a small 2-class subset of CIFAR-10 (`airplane` vs. `automobile`), so training is fast even on CPU. Because only the head is trained, this needs very little data/compute — the point of transfer learning.

In [ ]:
from torchvision import datasets as tv_datasets, transforms as tv_transforms

resnet_transform = tv_transforms.Compose([
    tv_transforms.Resize((128, 128)),
    tv_transforms.ToTensor(),
    tv_transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

cifar_train = tv_datasets.CIFAR10(root="./data", train=True, download=True, transform=resnet_transform)
cifar_test = tv_datasets.CIFAR10(root="./data", train=False, download=True, transform=resnet_transform)

# Keep only 2 classes: 0 = airplane, 1 = automobile
classes_of_interest = [0, 1]

def filter_classes(dataset, classes, max_per_class=200):
    indices = []
    counts = {c: 0 for c in classes}
    for idx, (_, label) in enumerate(dataset):
        if label in classes and counts[label] < max_per_class:
            indices.append(idx)
            counts[label] += 1
        if all(v >= max_per_class for v in counts.values()):
            break
    return Subset(dataset, indices)

train_subset2 = filter_classes(cifar_train, classes_of_interest, max_per_class=150)
test_subset2 = filter_classes(cifar_test, classes_of_interest, max_per_class=50)

train_loader2 = DataLoader(train_subset2, batch_size=16, shuffle=True)
test_loader2 = DataLoader(test_subset2, batch_size=32, shuffle=False)

transfer_model = build_transfer_model(num_classes=2).to(device)

# Remap CIFAR labels {0, 1} -> {0, 1} for the 2-way classifier
label_map = {c: i for i, c in enumerate(classes_of_interest)}

optimizer2 = optim.Adam(filter(lambda p: p.requires_grad, transfer_model.parameters()), lr=1e-3)
criterion2 = nn.CrossEntropyLoss()

n_epochs2 = 2
for epoch in range(n_epochs2):
    transfer_model.train()
    running_loss = 0.0
    for images, labels in train_loader2:
        labels = torch.tensor([label_map[l.item()] for l in labels])
        images, labels = images.to(device), labels.to(device)
        optimizer2.zero_grad()
        outputs = transfer_model(images)
        loss = criterion2(outputs, labels)
        loss.backward()
        optimizer2.step()
        running_loss += loss.item() * images.size(0)
    print(f"Epoch {epoch + 1}/{n_epochs2} - train loss: {running_loss / len(train_subset2):.4f}")

transfer_model.eval()
correct, total = 0, 0
with torch.no_grad():
    for images, labels in test_loader2:
        labels = torch.tensor([label_map[l.item()] for l in labels])
        images, labels = images.to(device), labels.to(device)
        preds = transfer_model(images).argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

print(f"\nTest accuracy on {total} held-out images (airplane vs automobile): {100 * correct / total:.2f}%")


---
## Exercise 5 — Neural style transfer (content loss + Gram-matrix style loss)

**Recap (slides):** "inverting" a classifier CNN lets us *generate* images. **Neural style transfer** re-renders a content image $\mathbf{x}_c$ in the style of a style image $\mathbf{x}_s$ by optimizing a new image $\mathbf{x}$ to minimize

$$\mathcal{L}(\mathbf{x}) = \lambda_c\,\mathcal{L}_{\text{content}}(\mathbf{x},\mathbf{x}_c) + \lambda_s\,\mathcal{L}_{\text{style}}(\mathbf{x},\mathbf{x}_s)$$

using the features $\boldsymbol\phi_\ell(\cdot)$ of a pretrained CNN (VGG19):

- **Content loss** at layer $\ell$: $\mathcal{L}_{\text{content}} = \frac{1}{C_\ell H_\ell W_\ell}\lVert\boldsymbol\phi_\ell(\mathbf{x}) - \boldsymbol\phi_\ell(\mathbf{x}_c)\rVert_2^2$
- **Gram matrix** at layer $\ell$ (captures feature *co-occurrence*, i.e. style): $G_\ell(\mathbf{x})_{c,d} = \frac{1}{H_\ell W_\ell}\sum_{h,w}\boldsymbol\phi_\ell(\mathbf{x})_{h,w,c}\,\boldsymbol\phi_\ell(\mathbf{x})_{h,w,d}$
- **Style loss** at layer $\ell$: $\mathcal{L}_{\text{style}}^\ell = \lVert\mathbf{G}_\ell(\mathbf{x}) - \mathbf{G}_\ell(\mathbf{x}_s)\rVert_F^2$, summed over a set of layers

**Your task:** implement `gram_matrix`, `content_loss`, and `style_loss` below. (The VGG19 feature extractor, image loading, and optimization loop are provided.)


In [ ]:
def gram_matrix(feat):
    """
    feat: tensor of shape (1, C, H, W)
    returns: (C, C) Gram matrix, normalized by (H * W) as in the slides
    """
    # TODO: reshape `feat` to (C, H*W), then compute f @ f.T, and divide
    #       by (H * W)
    raise NotImplementedError("Implement gram_matrix")


def content_loss(feat_x, feat_content):
    """Mean squared error between two feature maps of the same shape."""
    # TODO: return the mean squared difference between feat_x and feat_content
    raise NotImplementedError("Implement content_loss")


def style_loss(feat_x, feat_style):
    """Squared Frobenius norm between the Gram matrices of two feature maps."""
    # TODO: compute the Gram matrix of feat_x and of feat_style (using your
    #       gram_matrix function above), then return the sum of squared
    #       differences between them
    raise NotImplementedError("Implement style_loss")


**Optimization loop** (given): we use two sample photographs from `scikit-image` as the content and style images (so nothing needs to be downloaded from the internet), extract features from a few VGG19 layers, and optimize the pixels of a copy of the content image to minimize your `content_loss` + `style_loss`. This runs for a modest number of iterations so it finishes in a reasonable time on CPU; expect a "stylized" rather than a fully polished result.

In [ ]:
import torchvision.models as tv_models
import torchvision.transforms as T
from skimage import data as skimage_data
from PIL import Image

# --- Load sample content & style images (bundled with scikit-image, no download needed) ---
content_pil = Image.fromarray(skimage_data.astronaut())
style_pil = Image.fromarray(skimage_data.coffee())

img_size = 128
loader = T.Compose([
    T.Resize((img_size, img_size)),
    T.ToTensor(),
])

def load_image(pil_img):
    return loader(pil_img).unsqueeze(0).to(device)

content_img = load_image(content_pil)
style_img = load_image(style_pil)

# --- Pretrained VGG19 feature extractor (frozen) ---
vgg = tv_models.vgg19(weights=tv_models.VGG19_Weights.DEFAULT).features.to(device).eval()
for p in vgg.parameters():
    p.requires_grad_(False)

content_layers = {"21": "content"}   # relu4_2
style_layers = {"0": "style1", "5": "style2", "10": "style3", "19": "style4"}

def extract_features(x, layers):
    feats = {}
    for name, layer in vgg._modules.items():
        x = layer(x)
        if name in layers:
            feats[layers[name]] = x
    return feats

with torch.no_grad():
    content_targets = extract_features(content_img, content_layers)
    style_targets = extract_features(style_img, style_layers)

# --- Optimize a copy of the content image ---
generated = content_img.clone().requires_grad_(True)
optimizer3 = optim.Adam([generated], lr=0.03)

lambda_c, lambda_s = 1.0, 1e4
n_steps = 150

for step in range(n_steps):
    optimizer3.zero_grad()
    gen_content_feats = extract_features(generated, content_layers)
    gen_style_feats = extract_features(generated, style_layers)

    c_loss = content_loss(gen_content_feats["content"], content_targets["content"])
    s_loss = sum(style_loss(gen_style_feats[k], style_targets[k]) for k in style_layers.values())

    total_loss = lambda_c * c_loss + lambda_s * s_loss
    total_loss.backward()
    optimizer3.step()

    if step % 30 == 0 or step == n_steps - 1:
        print(f"step {step:4d} | content {c_loss.item():.4f} | style {s_loss.item():.6f} | total {total_loss.item():.4f}")

def to_image(tensor):
    img = tensor.detach().cpu().squeeze(0).clamp(0, 1).permute(1, 2, 0).numpy()
    return img

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(to_image(content_img)); axes[0].set_title("Content")
axes[1].imshow(to_image(style_img)); axes[1].set_title("Style")
axes[2].imshow(to_image(generated)); axes[2].set_title("Generated")
for ax in axes:
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
plt.show()
